<div align="center">

# 🎙️ Fine-Tuning Qwen3-ASR for Moroccan Darija
### A Hands-On Tutorial for Engineering Students

---

**Based on the MoulSot project** — *Building a Moroccan Darija Speech Recognition System*  
Original blog post: [Building MoulSot on Hugging Face](https://huggingface.co/blog/abdeljalilELmajjodi/moulsot)

---

</div>

## 🎯 What You Will Learn

By the end of this notebook, you will be able to:

| # | Skill |
|---|-------|
| 1 | Understand what ASR (Automatic Speech Recognition) is and how modern models work |
| 2 | Load and explore a real-world speech dataset from Hugging Face |
| 3 | Prepare audio data in the format required by Qwen3-ASR |
| 4 | Fine-tune a pre-trained ASR model on a target language/dialect |
| 5 | Evaluate ASR quality using WER and CER metrics |
| 6 | Run inference on new audio samples |

## 📖 Background: The MoulSot Project

**Moroccan Darija** is spoken by 30+ million people but is severely under-resourced in speech technology. Unlike Modern Standard Arabic, Darija blends Arabic roots with Amazigh, French, and Spanish influences — making it a unique and challenging ASR target.

The **MoulSot** project built a full data pipeline:
1. Crawled ~1,500 hours of Darija speech from YouTube
2. Filtered & scored audio quality using multiple AI models
3. Transcribed the best 80 hours using Gemini 2.5 Pro
4. **Fine-tuned `Qwen3-ASR-0.6B`** — which is what we'll do in this notebook!

> ⚡ **In this tutorial:** We use only **1,000 training samples** to keep training fast and approachable. In production, the full dataset was ~80 hours.

---

## ⚙️ Runtime Setup

**Before running this notebook:**
1. Go to `Runtime` → `Change runtime type`
2. Select **GPU** (T4 is sufficient for this tutorial)
3. Click Save

You should see a GPU listed when you run the environment check below.

---
## 📦 Section 1 — Environment Check & Package Installation

Let's start by verifying our GPU is available and installing all required libraries.

In [8]:
# ── Check GPU availability ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("⚠️  No GPU detected. Please enable GPU in Runtime > Change runtime type.")
    print("   Training will be very slow on CPU-only.")

Sat May 16 11:23:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# ── Install required packages ───────────────────────────────────────────────
# This cell installs everything we need. It may take 3-5 minutes.

print("📥 Installing core packages...")
!pip install -q \
    transformers \
    datasets \
    accelerate \
    soundfile \
    librosa \
    jiwer \
    tqdm \
    huggingface_hub

!pip install qwen_asr

📥 Installing core packages...


In [10]:
# ── Clone the official Qwen3-ASR fine-tuning repository ────────────────────
# The official repo provides the SFT training script we'll use

import os

if not os.path.exists('Qwen3-ASR'):
    print("📂 Cloning Qwen3-ASR official repository...")
    !git clone https://github.com/QwenLM/Qwen3-ASR.git
    print("✅ Repository cloned!")
else:
    print("✅ Repository already exists.")

# Install the fine-tuning requirements from the repo
if os.path.exists('Qwen3-ASR/finetuning/requirements.txt'):
    print("📥 Installing fine-tuning requirements...")
    !pip install -q -r Qwen3-ASR/finetuning/requirements.txt
    print("✅ Fine-tuning requirements installed!")

✅ Repository already exists.


---
## 🧠 Section 2 — Understanding Qwen3-ASR

### What is ASR?

**Automatic Speech Recognition (ASR)** is the task of converting spoken audio into text. Modern ASR systems use deep learning, typically with an **encoder-decoder architecture**:

```
🎤 Audio waveform
       │
       ▼
  [ Encoder ]  →  learns acoustic features (phonemes, sounds)
       │
       ▼
  [ Decoder ]  →  generates text tokens auto-regressively
       │
       ▼
📝 "مرحبا، كيف حالك؟"
```

### Why Qwen3-ASR?

[Qwen3-ASR](https://huggingface.co/Qwen/Qwen3-ASR-0.6B) is a model from Alibaba Cloud (Qwen Team) that:
- Follows a **Whisper-style** architecture (proven encoder-decoder for ASR)
- Was **pre-trained on massively multilingual speech** data, including Arabic
- Comes in two sizes: **0.6B** (fast) and **1.7B** (more accurate) — we use **1.7B**
- Uses a special text format: `language Arabic<asr_text>your transcript here`

### Why Fine-Tune?

Pre-training covers Modern Standard Arabic, but **Moroccan Darija** is very different:
- Different vocabulary (French & Amazigh loanwords)
- Code-switching mid-sentence (Darija + French)
- Regional accent variation
- Different phonological rules

Fine-tuning on Darija data **adapts the model** to these specificities without training from scratch.

In [11]:
# ── Verify PyTorch and CUDA setup ───────────────────────────────────────────
import torch

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU name         : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory (GB)  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
else:
    device = torch.device('cpu')
    print("⚠️  Running on CPU — training will be slow!")

print(f"\n✅ Using device: {device}")

PyTorch version  : 2.10.0+cu128
CUDA available   : True
GPU name         : Tesla T4
GPU memory (GB)  : 15.6

✅ Using device: cuda


---
## 📊 Section 3 — Loading the MoulSot Dataset

We'll load the **MoulSot-Full** dataset from Hugging Face. This is the curated 80-hour Moroccan Darija corpus built by the MoulSot project.

The dataset has two configurations:
- **`100-gt-2.5`**: Training split — samples with PESQ score > 2.5 (highest quality)
- **`default`**: Test split — held-out evaluation samples

> 🔬 **For this tutorial**: We'll take only **1,000 training samples** to keep things fast. The full dataset has tens of thousands of samples.

In [12]:
# ── Load dataset from Hugging Face ──────────────────────────────────────────
from datasets import load_dataset
import pandas as pd

print("📥 Loading MoulSot-Full training split (config: 100-gt-2.5)...")
print("   This selects samples with PESQ audio quality score > 2.5")

# Load the high-quality training config
# If the dataset requires authentication, you may need to add token=YOUR_HF_TOKEN
train_ds_full = load_dataset(
    "atlasia/MoulSot-Full-80",
    split="train",
    trust_remote_code=True
)

print(f"✅ Full training set loaded: {len(train_ds_full):,} samples")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'atlasia/MoulSot-Full-80' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'atlasia/MoulSot-Full-80' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📥 Loading MoulSot-Full training split (config: 100-gt-2.5)...
   This selects samples with PESQ audio quality score > 2.5
✅ Full training set loaded: 1,000 samples


In [13]:
# ── Load test split ─────────────────────────────────────────────────────────
print("📥 Loading MoulSot-Full test split...")

test_ds_full = load_dataset(
    "atlasia/MoulSot-Full-80",
    split="test",
    trust_remote_code=True
)

print(f"✅ Full test set loaded: {len(test_ds_full):,} samples")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'atlasia/MoulSot-Full-80' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'atlasia/MoulSot-Full-80' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📥 Loading MoulSot-Full test split...
✅ Full test set loaded: 200 samples


In [14]:
# ── Subsample to 1k training examples ──────────────────────────────────────
# For this tutorial we only use 1,000 samples to keep training fast.
# In the MoulSot paper they used the full ~80 hours.

NUM_TRAIN_SAMPLES = 1000
NUM_TEST_SAMPLES  = 200   # keep test small too

# Shuffle with a fixed seed for reproducibility, then select
train_ds = train_ds_full.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES))
test_ds  = test_ds_full.shuffle(seed=42).select(range(min(NUM_TEST_SAMPLES, len(test_ds_full))))

print(f"✅ Subsampled training set : {len(train_ds):,} samples")
print(f"✅ Subsampled test set     : {len(test_ds):,} samples")
print()
print("💡 Tip: Increase NUM_TRAIN_SAMPLES to train on more data (at the cost of time).")

✅ Subsampled training set : 1,000 samples
✅ Subsampled test set     : 200 samples

💡 Tip: Increase NUM_TRAIN_SAMPLES to train on more data (at the cost of time).


In [15]:
# ── Explore the dataset structure ───────────────────────────────────────────
print("📋 Dataset Features:")
print(train_ds.features)
print()

# Look at a single example
sample = train_ds[0]
print("📌 Example sample (first item):")
for key, val in sample.items():
    if key == 'audio':
        print(f"  audio:")
        print(f"    sampling_rate : {val['sampling_rate']} Hz")
        print(f"    array shape   : {val['array'].shape}")
        duration = len(val['array']) / val['sampling_rate']
        print(f"    duration      : {duration:.2f} seconds")
    else:
        print(f"  {key:20s}: {val}")

📋 Dataset Features:
{'id': Value('string'), 'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'sample_rate': Value('int64'), 'n_channels': Value('int64'), 'pesq_hyp': Value('float64'), 'stoi_hyp': Value('float64'), 'si_sdr_hyp': Value('float64'), 'text': Value('string'), 'channel': Value('string'), 'duration': Value('float64')}

📌 Example sample (first item):
  id                  : @BouachrineTaoufik_57_192
  audio:
    sampling_rate : 16000 Hz
    array shape   : (12800,)
    duration      : 0.80 seconds
  sample_rate         : 16000
  n_channels          : 1
  pesq_hyp            : 4.17578125
  stoi_hyp            : 0.99560546875
  si_sdr_hyp          : 28.578125
  text                : وكذلك
  channel             : @BouachrineTaoufik
  duration            : 0.8


In [16]:
# ── Display a few transcripts to understand the data ────────────────────────
print("📝 Sample transcripts from the dataset:")
print("-" * 60)

# Try common column names for transcripts
text_col = None
for candidate in ['text', 'transcription', 'sentence', 'transcript']:
    if candidate in train_ds.features:
        text_col = candidate
        break

if text_col:
    for i in range(5):
        s = train_ds[i]
        dur = len(s['audio']['array']) / s['audio']['sampling_rate']
        print(f"[{i+1}] Duration : {dur:.1f}s")
        print(f"    Transcript: {s[text_col]}")
        print()
else:
    print("Could not find transcript column. Available columns:", train_ds.column_names)

📝 Sample transcripts from the dataset:
------------------------------------------------------------
[1] Duration : 0.8s
    Transcript: وكذلك

[2] Duration : 7.0s
    Transcript: Lifetime customer value هي شحال ما حدو معاك شحال كت هادا مقسومة على customer acquisition cost بشحال كيتقام عليك هاداك الكليان

[3] Duration : 5.6s
    Transcript: وتصلح الخطأ ديالك بالنسبة ليا التهور زوين حيت يمكن إلا ماكانش داك التهور بعض المرات

[4] Duration : 3.8s
    Transcript: كان له لقاء مع الفريق اول السيد Son Sen

[5] Duration : 2.1s
    Transcript: وعندما منعت فرنسا النقاب



In [17]:
# ── Listen to a sample audio clip (Colab-compatible playback) ───────────────
from IPython.display import Audio, display
import numpy as np

sample_idx = 0
sample = train_ds[sample_idx]
audio_array = sample['audio']['array']
sample_rate  = sample['audio']['sampling_rate']

print(f"🎵 Playing sample #{sample_idx}")
if text_col:
    print(f"📝 Transcript: {sample[text_col]}")
print(f"⏱  Duration  : {len(audio_array)/sample_rate:.2f} seconds")
print(f"📡 Sample rate: {sample_rate} Hz")

display(Audio(audio_array, rate=sample_rate))

🎵 Playing sample #0
📝 Transcript: وكذلك
⏱  Duration  : 0.80 seconds
📡 Sample rate: 16000 Hz


---
## 🔧 Section 4 — Data Preparation

Qwen3-ASR requires a **very specific data format**. Let's understand it and prepare our data.

### The Qwen3-ASR JSONL Format

Each training example must be a JSON line with two fields:

```json
{"audio": "/absolute/path/to/clip.wav", "text": "language Arabic<asr_text>نص المحادثة هنا"}
```

Key points:
- **`audio`**: Absolute path to a **16 kHz mono WAV** file
- **`text`**: Must start with `language Arabic<asr_text>` followed by the transcript
- The `language Arabic` prefix is the **language hint** (tells the model which language to use)
- `<asr_text>` is a **separator tag** the model uses internally

> 🔑 If the text format is wrong, the model will either fail to train or learn incorrect behavior.

In [18]:
# ── Setup directories ────────────────────────────────────────────────────────
import os

# Create all needed directories
DIRS = {
    'data'       : 'data',
    'train_wavs' : 'data/wavs/train',
    'test_wavs'  : 'data/wavs/test',
    'output'     : 'model_output'
}

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✅ Created: {path}")

✅ Created: data
✅ Created: data/wavs/train
✅ Created: data/wavs/test
✅ Created: model_output


In [19]:
# ── Data preparation function ────────────────────────────────────────────────
# This function converts a HuggingFace dataset split into:
#   1. WAV files at 16 kHz (saved to disk)
#   2. A JSONL manifest file for Qwen3-ASR

import json
import soundfile as sf
import numpy as np
import librosa
from tqdm import tqdm


def resample_audio(audio_array: np.ndarray, orig_sr: int, target_sr: int = 16000) -> np.ndarray:
    """Resample audio array from orig_sr to target_sr."""
    if orig_sr == target_sr:
        return audio_array
    return librosa.resample(audio_array, orig_sr=orig_sr, target_sr=target_sr)


def prepare_dataset_split(
    dataset,
    split_name: str,
    wav_dir: str,
    text_col: str,
    target_sr: int = 16000,
) -> str:
    """
    Save WAV files + write JSONL manifest for one dataset split.

    Returns:
        Path to the generated .jsonl file.
    """
    os.makedirs(wav_dir, exist_ok=True)
    jsonl_path = os.path.join('data', f'{split_name}.jsonl')

    skipped = 0
    written = 0

    with open(jsonl_path, 'w', encoding='utf-8') as f_out:
        for idx, example in enumerate(tqdm(dataset, desc=f'Preparing {split_name}')):

            # ── Get transcript ──────────────────────────────────────────────
            try:
              transcript = example.get(text_col, '').strip()
            except:
              skipped += 1
              continue

            # ── Get audio ──────────────────────────────────────────────────
            audio_info = example['audio']
            audio_array = audio_info['array'].astype(np.float32)
            orig_sr     = audio_info['sampling_rate']

            # ── Resample to 16 kHz (Qwen3-ASR requirement) ─────────────────
            audio_16k = resample_audio(audio_array, orig_sr, target_sr)

            # ── Skip very short or very long clips ─────────────────────────
            duration = len(audio_16k) / target_sr
            if duration < 0.5 or duration > 30.0:
                skipped += 1
                continue

            # ── Save WAV file ──────────────────────────────────────────────
            wav_filename = f'{split_name}_{idx:06d}.wav'
            wav_path     = os.path.abspath(os.path.join(wav_dir, wav_filename))
            sf.write(wav_path, audio_16k, target_sr)

            # ── Build Qwen3-ASR JSONL entry ────────────────────────────────
            # IMPORTANT: The text MUST follow this exact format:
            #   "language Arabic<asr_text>{transcript}"
            qwen_text = f'language Arabic<asr_text>{transcript}'

            entry = {
                'audio': wav_path,
                'text' : qwen_text
            }
            f_out.write(json.dumps(entry, ensure_ascii=False) + '\n')
            written += 1

    print(f"\n✅ {split_name}: {written} samples written, {skipped} skipped")
    print(f"   JSONL saved to: {jsonl_path}")
    return jsonl_path


print("✅ Data preparation functions defined.")

✅ Data preparation functions defined.


In [20]:
# ── Prepare the TRAINING split ──────────────────────────────────────────────
# Saves 1,000 WAV files + writes data/train.jsonl

# Auto-detect the transcript column name
text_col = None
for candidate in ['text', 'transcription', 'sentence', 'transcript']:
    if candidate in train_ds.features:
        text_col = candidate
        break

if text_col is None:
    raise ValueError(f"Cannot find transcript column. Available: {train_ds.column_names}")

print(f"Using transcript column: '{text_col}'\n")

train_jsonl = prepare_dataset_split(
    dataset   = train_ds,
    split_name= 'train',
    wav_dir   = DIRS['train_wavs'],
    text_col  = text_col,
)

Using transcript column: 'text'



Preparing train: 100%|██████████| 1000/1000 [00:31<00:00, 31.63it/s]


✅ train: 998 samples written, 2 skipped
   JSONL saved to: data/train.jsonl


In [21]:
# ── Prepare the TEST split ──────────────────────────────────────────────────
# Saves test WAV files + writes data/test.jsonl

test_jsonl = prepare_dataset_split(
    dataset   = test_ds,
    split_name= 'test',
    wav_dir   = DIRS['test_wavs'],
    text_col  = text_col,
)

Preparing test: 100%|██████████| 200/200 [00:02<00:00, 88.29it/s]


✅ test: 196 samples written, 4 skipped
   JSONL saved to: data/test.jsonl


In [22]:
# ── Inspect the generated JSONL files ───────────────────────────────────────
print("📄 First 3 lines of data/train.jsonl:")
print("-" * 70)
with open(train_jsonl, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        entry = json.loads(line)
        print(f"Line {i+1}:")
        print(f"  audio : {entry['audio']}")
        print(f"  text  : {entry['text'][:100]}..." if len(entry['text']) > 100 else f"  text  : {entry['text']}")
        print()
        if i >= 2:
            break

# Count lines
with open(train_jsonl) as f:
    n_train = sum(1 for _ in f)
with open(test_jsonl) as f:
    n_test = sum(1 for _ in f)

print(f"✅ Total training samples : {n_train}")
print(f"✅ Total test samples     : {n_test}")

📄 First 3 lines of data/train.jsonl:
----------------------------------------------------------------------
Line 1:
  audio : /content/data/wavs/train/train_000000.wav
  text  : language Arabic<asr_text>وكذلك

Line 2:
  audio : /content/data/wavs/train/train_000001.wav
  text  : language Arabic<asr_text>Lifetime customer value هي شحال ما حدو معاك شحال كت هادا مقسومة على custome...

Line 3:
  audio : /content/data/wavs/train/train_000002.wav
  text  : language Arabic<asr_text>وتصلح الخطأ ديالك بالنسبة ليا التهور زوين حيت يمكن إلا ماكانش داك التهور بع...

✅ Total training samples : 998
✅ Total test samples     : 196


---
## 🏋️ Section 5 — Fine-Tuning Qwen3-ASR

### How Fine-Tuning Works

Fine-tuning starts from a **pre-trained checkpoint** and continues training on your specific data:

```
Qwen3-ASR-1.7B (pre-trained, multilingual)
         │
         │  Fine-tune on MoulSot data (Moroccan Darija)
         │  • Feed (audio, transcript) pairs
         │  • Compute cross-entropy loss
         │  • Update weights via backprop + AdamW
         ▼
Qwen3-ASR-1.7B-MoulSot (Darija-specialized)
```

### Key Hyperparameters

| Parameter | Our Value | Explanation |
|-----------|-----------|-------------|
| **Epochs** | 3 | Number of full passes over training data |
| **Batch size** | 4 | Samples processed together per step |
| **Gradient accumulation** | 2 | Simulate larger batch without more memory |
| **Learning rate** | 5e-5 | How fast weights update |
| **LR scheduler** | cosine | Gradually reduces LR following a cosine curve |
| **Warmup ratio** | 0.05 | First 5% of steps slowly ramp up the LR |
| **Precision** | bf16 | Half-precision to save GPU memory |

> 🔑 We use 3 epochs instead of 5 (from the paper) and batch=4 instead of 8 to fit in Colab's T4 GPU memory.

In [23]:
# ── Verify the SFT script exists ─────────────────────────────────────────────
sft_script = 'Qwen3-ASR/finetuning/qwen3_asr_sft.py'

if os.path.exists(sft_script):
    print(f"✅ Fine-tuning script found: {sft_script}")
else:
    print(f"⚠️  Script not found at {sft_script}")
    print("   Trying to list available scripts...")
    if os.path.exists('Qwen3-ASR'):
        for root, dirs, files in os.walk('Qwen3-ASR'):
            for f in files:
                if f.endswith('.py'):
                    print(f"   Found: {os.path.join(root, f)}")
    else:
        print("   Repository not found. Re-run the clone cell above.")

✅ Fine-tuning script found: Qwen3-ASR/finetuning/qwen3_asr_sft.py


In [24]:
!nvidia-smi

Sat May 16 11:24:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
# ── Build the fine-tuning command ────────────────────────────────────────────
import os

MODEL_NAME  = 'Qwen/Qwen3-ASR-0.6B'
OUTPUT_DIR  = os.path.abspath(DIRS['output'])
TRAIN_JSONL = os.path.abspath(train_jsonl)
TEST_JSONL  = os.path.abspath(test_jsonl)

# ─── Hyperparameters ────────────────────────────────────────────────────────
EPOCHS        = 3      # full passes over data
BATCH_SIZE    = 4      # per-GPU batch (fits T4 16GB)
GRAD_ACC      = 2      # gradient accumulation steps → effective batch = 8
LR            = 5e-6   # learning rate
WARMUP_RATIO  = 0.05   # 5% warmup
LR_SCHEDULER  = 'cosine'
NUM_GPUS      = 1      # Colab has 1 GPU


print()
print("⏱  Estimated training time on T4 GPU:")
print(f"   ~{EPOCHS * 1000 * 10 / 3600:.1f} hours for {NUM_TRAIN_SAMPLES} samples × {EPOCHS} epochs")
print("   (Actual time depends on GPU speed and average clip length)")


⏱  Estimated training time on T4 GPU:
   ~8.3 hours for 1000 samples × 3 epochs
   (Actual time depends on GPU speed and average clip length)


In [27]:
file_path = "Qwen3-ASR/finetuning/qwen3_asr_sft.py"

with open(file_path, "r") as f:
    code = f.read()

# 1. Revert my previous messy patch
code = code.replace("TrainingArguments(fp16=False, ", "TrainingArguments(")

# 2. Safely replace the actual line causing the mixed-precision crash
code = code.replace("fp16=not use_bf16,", "fp16=False,")

with open(file_path, "w") as f:
    f.write(code)

print("Duplicate keyword fixed and repo properly patched! Try running again.")

Duplicate keyword fixed and repo properly patched! Try running again.


In [28]:
!python Qwen3-ASR/finetuning/qwen3_asr_sft.py \
    --model_path {MODEL_NAME} \
    --train_file {TRAIN_JSONL} \
    --eval_file  {TEST_JSONL} \
    --output_dir {OUTPUT_DIR} \
    --epochs 1 \
    --batch_size 4 \
    --grad_acc 1 \
    --lr {LR} \
    --warmup_ratio {WARMUP_RATIO} \
    --lr_scheduler_type {LR_SCHEDULER} \
    --num_workers 1

2026-05-16 11:25:59.012771: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 6.19kB [00:00, 14.0MB/s]
model.safetensors: 100% 1.88G/1.88G [00:30<00:00, 60.8MB/s]
generation_config.json: 100% 142/142 [00:00<00:00, 1.13MB/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
preprocessor_config.json: 100% 330/330 [00:00<00:00, 2.85MB/s]
tokenizer_config.json: 12.5kB [00:00, 54.4MB/s]
vocab.json: 2.78MB [00:00, 72.9MB/s]
merges.txt: 1.67MB [00:00, 87.3MB/s]
chat_template.json: 1.16kB [00:00, 1.17MB/s]
Generating train split: 998 examples [00:00, 119488.34 examples/s]
Generating validation split: 196 examples [00:00, 147247.64 examples/s]
Map: 100% 998/998 

In [29]:
# ── Check that checkpoints were saved ────────────────────────────────────────
print("📁 Contents of model output directory:")
!ls -lh {OUTPUT_DIR}/ 2>/dev/null || echo "Output directory not found yet."

# Find the latest checkpoint
import glob
checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')))

if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f"\n✅ Latest checkpoint: {latest_ckpt}")
    print("   Contents:")
    !ls {latest_ckpt}/
else:
    print("\n⚠️  No checkpoints found. Training may not have completed.")

📁 Contents of model output directory:
total 8.0K
drwxr-xr-x 2 root root 4.0K May 16 11:30 checkpoint-200
drwxr-xr-x 2 root root 4.0K May 16 11:34 checkpoint-250

✅ Latest checkpoint: /content/model_output/checkpoint-250
   Contents:
added_tokens.json	optimizer.pt		 tokenizer.json
config.json		rng_state.pth		 trainer_state.json
generation_config.json	scheduler.pt		 training_args.bin
merges.txt		special_tokens_map.json  vocab.json
model.safetensors	tokenizer_config.json


---
## 📈 Section 6 — Evaluation

### ASR Evaluation Metrics

We evaluate our fine-tuned model with three metrics:

#### 1. WER — Word Error Rate
$$\text{WER} = \frac{\text{Substitutions} + \text{Deletions} + \text{Insertions}}{\text{Total reference words}}$$

- WER = 0.0 → perfect transcription
- WER = 1.0 → completely wrong
- Lower is better! State-of-the-art models achieve WER < 0.10 on well-resourced languages.

#### 2. CER — Character Error Rate
Same idea but at the **character level** instead of word level. For Arabic script, CER is often more meaningful because word boundaries can be ambiguous.

#### 3. RTF — Real-Time Factor
$$\text{RTF} = \frac{\text{Inference time}}{\text{Audio duration}}$$

RTF < 1.0 means the model is **faster than real-time** — practical for deployment.

In [30]:
import os, glob, torch
from transformers import AutoProcessor
from qwen_asr import Qwen3ASRModel

# 1. Find the latest checkpoint
base_id = 'Qwen/Qwen3-ASR-0.6B'  # Make sure this matches the base model you trained on!
ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')), key=os.path.getmtime)
model_path = ckpts[-1] if ckpts else base_id

# --- THE FIX ---
# Inject the missing processor files into the checkpoint directory so Qwen3ASRModel can read them
if ckpts:
    print(f"🔧 Patching missing processor files into {model_path}...")
    temp_processor = AutoProcessor.from_pretrained(base_id)
    temp_processor.save_pretrained(model_path)
# ---------------

print(f"{'🔍 Loading fine-tuned checkpoint' if ckpts else '⚠️ No checkpoint found, loading base'}: {model_path}")

# 2. Load Model natively via Qwen3ASRModel
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
asr = Qwen3ASRModel.from_pretrained(
    model_path,
    dtype=dtype,
    device_map=device,
    max_inference_batch_size=16,
)

print(f"✅ Ready! Model loaded on {device}.")

🔧 Patching missing processor files into /content/model_output/checkpoint-250...
🔍 Loading fine-tuned checkpoint: /content/model_output/checkpoint-250
✅ Ready! Model loaded on cuda.


In [31]:
import json
import time
import numpy as np
import soundfile as sf
import jiwer
from tqdm import tqdm

# Helper to normalize text so casing/spaces don't ruin our WER scores
def normalise_qwen_output(text: str) -> str:
    if not text:
        return ""
    return str(text).strip().lower()

# Helper to calculate Word Error Rate and Character Error Rate
def compute_metrics(refs, hyps):
    # Filter out empty references to prevent jiwer from crashing
    valid_refs, valid_hyps = [], []
    for r, h in zip(refs, hyps):
        if r.strip():
            valid_refs.append(r)
            valid_hyps.append(h)

    if not valid_refs:
        return {"wer": 1.0, "cer": 1.0}

    wer = jiwer.wer(valid_refs, valid_hyps)
    cer = jiwer.cer(valid_refs, valid_hyps)
    return {"wer": wer, "cer": cer}


# ── Run evaluation on the test set ────────────────────────────────────────────

print("📊 Preparing evaluation data...")

references   = []
hypotheses   = []
rtf_values   = []
results_list = []

# Load test JSONL (Assuming TEST_JSONL variable is defined in your notebook)
test_entries = []
with open(TEST_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        test_entries.append(json.loads(line.strip()))

n_test = len(test_entries)
EVAL_SAMPLES = min(50, n_test)
print(f"📊 Evaluating on {EVAL_SAMPLES} test samples...")

for i, entry in enumerate(tqdm(test_entries[:EVAL_SAMPLES], desc='Evaluating')):
    wav_path  = entry['audio']
    reference = normalise_qwen_output(entry['text'])

    # Load WAV for RTF (Real-Time Factor) calculation
    audio_array, sr = sf.read(wav_path)
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)  # stereo → mono

    duration = len(audio_array) / sr

    # Time the inference using Qwen3ASRModel
    t0 = time.time()

    # Pass the audio tuple exactly like in your Gradio Space
    results = asr.transcribe(
        audio=(audio_array.astype(np.float32), sr),
        language="Arabic"
    )

    # Safely extract text from the returned Result object
    raw_hypothesis = getattr(results[0], "text", "") if results else ""
    hypothesis = normalise_qwen_output(raw_hypothesis)

    elapsed = time.time() - t0

    rtf = elapsed / duration if duration > 0 else 0

    references.append(reference)
    hypotheses.append(hypothesis)
    rtf_values.append(rtf)
    results_list.append({
        'id'       : i,
        'reference': reference,
        'hypothesis': hypothesis,
        'duration' : round(duration, 2),
        'rtf'      : round(rtf, 3),
    })

# ── Compute aggregate metrics ─────────────────────────────────────────────────
metrics = compute_metrics(references, hypotheses)

print("\n" + "=" * 50)
print("📊 EVALUATION RESULTS")
print("=" * 50)
print(f"   WER  : {metrics['wer']:.4f}  ({metrics['wer']*100:.1f}%)")
print(f"   CER  : {metrics['cer']:.4f}  ({metrics['cer']*100:.1f}%)")
print(f"   RTF  : {np.mean(rtf_values):.3f}  (mean)")
print("=" * 50 + "\n")

if metrics['wer'] is not None:
    if metrics['wer'] < 0.3:
        print("✅ Great WER! The model learned Darija well.")
    elif metrics['wer'] < 0.6:
        print("📈 Decent WER. More data / epochs would improve it.")
    else:
        print("📉 High WER — expected with limited training samples.")
        print("   Try training with more data for better results.")

📊 Preparing evaluation data...
📊 Evaluating on 50 test samples...


Evaluating: 100%|██████████| 50/50 [21:47<00:00, 26.15s/it]


📊 EVALUATION RESULTS
   WER  : 1.0000  (100.0%)
   CER  : 1.0000  (100.0%)
   RTF  : 12.805  (mean)

📉 High WER — expected with limited training samples.
   Try training with more data for better results.


---
## 🧪 Section 9 — Exercises

Now that you've completed the tutorial, try these exercises to deepen your understanding:

---

### 🟢 Exercise 1 — Effect of Training Data Size

Retrain the model with different amounts of data and compare WER:

| Config | Samples | Expected WER trend |
|--------|---------|--------------------|
| Tiny   | 100     | Very high |
| Small  | 500     | High |
| Medium | 1,000   | Moderate (this tutorial) |
| Large  | 5,000   | Better |

**What to do:** Change `NUM_TRAIN_SAMPLES` at the top and retrain. Record WER for each setting.

---

### 🟡 Exercise 2 — Hyperparameter Tuning

Try different learning rates and observe the effect on training loss:
- `LR = 1e-4` (too high — may diverge)
- `LR = 1e-6` (default — stable)
- `LR = 1e-8` (too low — slow convergence)

**What to do:** Modify the `LR` variable in Section 5 and retrain.



---
## 📚 Summary & Key Takeaways

<div style="background-color: #f0f7ff; padding: 20px; border-radius: 8px; border-left: 4px solid #2196F3;">

### What we did in this tutorial:

1. **Loaded** the MoulSot-Full dataset (Moroccan Darija speech corpus)
2. **Explored** audio/transcript statistics and listened to samples
3. **Prepared** data in Qwen3-ASR's required JSONL format (16 kHz WAV + `language Arabic<asr_text>` text)
4. **Fine-tuned** `Qwen3-ASR-1.7B` on 1,000 Darija samples using `torchrun`
5. **Evaluated** with WER, CER, and RTF metrics
6. **Ran inference** on new audio clips

### Key concepts learned:

| Concept | What you learned |
|---------|------------------|
| **ASR** | Encoder-decoder architecture for speech-to-text |
| **Fine-tuning** | Adapting a pre-trained model to a specific domain/language |
| **WER / CER** | Standard metrics to measure transcription accuracy |
| **RTF** | Practical metric for deployment speed |
| **Data format** | Why exact format matters for fine-tuning |
| **Language hints** | How to guide multilingual models |

### Want to go further?

- 📖 [Qwen3-ASR Paper & Model Card](https://huggingface.co/Qwen/Qwen3-ASR-1.7B)
- 🗂️ [MoulSot Dataset](https://huggingface.co/datasets/atlasia/MoulSot-Full)
- 🤗 [MoulSot Fine-tuned Model](https://huggingface.co/atlasia/moulsot.v0.3)
- 🎮 [Live Demo](https://huggingface.co/spaces/atlasia/MoulSot.v0.3)
- 📝 [Original Blog Post](https://huggingface.co/blog/abdeljalilELmajjodi/moulsot)

</div>

In [32]:
# ── Final summary printout ────────────────────────────────────────────────────
print("="*55)
print("  ✅  TUTORIAL COMPLETE!")
print("="*55)
print()
print("  Model     : Qwen3-ASR-1.7B (fine-tuned)")
print(f"  Dataset   : atlasia/MoulSot-Full ({NUM_TRAIN_SAMPLES} train samples)")
if metrics['wer'] is not None:
    print(f"  WER       : {metrics['wer']*100:.1f}%")
    print(f"  CER       : {metrics['cer']*100:.1f}%")
    print(f"  Mean RTF  : {np.mean(rtf_values):.3f}")
print()
print("  Files saved:")
print("    data/train.jsonl         - Training manifest")
print("    data/test.jsonl          - Test manifest")
print("    data/eval_details.tsv    - Per-sample results")
print("    duration_distribution.png")
print("    evaluation_results.png")
print()
print("  Good luck with the exercises! 🚀")
print("="*55)

  ✅  TUTORIAL COMPLETE!

  Model     : Qwen3-ASR-1.7B (fine-tuned)
  Dataset   : atlasia/MoulSot-Full (1000 train samples)
  WER       : 100.0%
  CER       : 100.0%
  Mean RTF  : 12.805

  Files saved:
    data/train.jsonl         - Training manifest
    data/test.jsonl          - Test manifest
    data/eval_details.tsv    - Per-sample results
    duration_distribution.png
    evaluation_results.png

  Good luck with the exercises! 🚀
